# Soma de Vetores em CUDA — Benchmark CPU vs GPU

**Disciplina:** Arquitetura de Computadores  
**Aluno:** Wésio Filho  
**Professor:** Newarney T. da Costa

Este notebook executa o benchmark do projeto usando os arquivos reais do repositório:

- `codigo/soma_cpu.c`
- `codigo/soma_gpu.cu`

Ele não reescreve versões simplificadas do código. Assim, o que é medido aqui é o mesmo código entregue no GitHub.


## 1. Verificar GPU ativa

No Colab, antes de rodar tudo, selecione:

`Runtime → Change runtime type → T4 GPU → Save`


In [ ]:
!nvidia-smi

## 2. Preparar o repositório

Se o notebook for aberto diretamente pelo GitHub no Colab, os arquivos do repo podem não estar no diretório atual. Esta célula verifica se `codigo/soma_cpu.c` existe; se não existir, clona o repositório público.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/wesioplayconstructor/cuda-soma-vetores.git'
REPO_DIR = Path('/content/cuda-soma-vetores')

if not Path('codigo/soma_cpu.c').exists():
    if not REPO_DIR.exists():
        print('Clonando repositório...')
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print('Arquivos do repositório já encontrados no diretório atual.')

print('Diretório atual:', Path.cwd())
print('Arquivos de código:')
print(' -', Path('codigo/soma_cpu.c').resolve())
print(' -', Path('codigo/soma_gpu.cu').resolve())

assert Path('codigo/soma_cpu.c').exists(), 'codigo/soma_cpu.c não encontrado'
assert Path('codigo/soma_gpu.cu').exists(), 'codigo/soma_gpu.cu não encontrado'


## 3. Compilar os programas

A versão CPU é compilada com `gcc` e a versão GPU com `nvcc`.


In [ ]:
!gcc -std=c11 -O2 -Wall -Wextra codigo/soma_cpu.c -o soma_cpu
!nvcc -O2 codigo/soma_gpu.cu -o soma_gpu

## 4. Conferir os primeiros resultados

Aqui rodamos com `N = 10` em modo normal, sem `--benchmark`, para mostrar os primeiros valores de `C`.

Como `A[i] = i` e `B[i] = 2*i`, então `C[i] = 3*i`.


In [ ]:
!./soma_cpu 10
!./soma_gpu 10

## 5. Benchmark

Agora rodamos cada versão 5 vezes para cada tamanho de vetor exigido no enunciado:

- 1.000 elementos
- 100.000 elementos
- 1.000.000 elementos
- 10.000.000 elementos

Os programas são executados com `--benchmark`, então a saída fica no formato:

```text
CPU 3.421000
GPU 0.842000
```


In [ ]:
import re
import subprocess
from pathlib import Path

import pandas as pd

TAMANHOS_N = [1000, 100000, 1000000, 10000000]
NUM_RUNS = 5
RESULTADOS_DIR = Path('resultados')
RESULTADOS_DIR.mkdir(exist_ok=True)

PADRAO_SAIDA = re.compile(r'^(CPU|GPU)\s+([0-9]+(?:\.[0-9]+)?)$')


def executar_benchmark(comando, rotulo_esperado):
    resultado = subprocess.run(comando, capture_output=True, text=True)

    if resultado.returncode != 0:
        raise RuntimeError(
            'Falha ao executar: ' + ' '.join(comando) + '\n'
            + 'STDOUT:\n' + resultado.stdout + '\n'
            + 'STDERR:\n' + resultado.stderr
        )

    saida = resultado.stdout.strip()
    match = PADRAO_SAIDA.match(saida)
    if not match:
        raise ValueError(
            'Saída inesperada do comando: ' + ' '.join(comando) + '\n'
            + 'Saída recebida:\n' + saida
        )

    rotulo, tempo = match.groups()
    if rotulo != rotulo_esperado:
        raise ValueError(f'Esperado rótulo {rotulo_esperado}, mas veio {rotulo}.')

    return float(tempo)


# Aquecimento: evita que a primeira chamada inclua custos extras de inicialização.
print('Aquecimento...')
executar_benchmark(['./soma_cpu', '1000', '--benchmark'], 'CPU')
executar_benchmark(['./soma_gpu', '1000', '--benchmark'], 'GPU')
print('Aquecimento concluído.\n')

resultados = []

for N in TAMANHOS_N:
    for run in range(1, NUM_RUNS + 1):
        tempo_cpu = executar_benchmark(['./soma_cpu', str(N), '--benchmark'], 'CPU')
        tempo_gpu = executar_benchmark(['./soma_gpu', str(N), '--benchmark'], 'GPU')
        speedup = tempo_cpu / tempo_gpu if tempo_gpu > 0 else float('inf')

        resultados.append({
            'N': N,
            'run': run,
            'tempo_cpu_ms': tempo_cpu,
            'tempo_gpu_ms': tempo_gpu,
            'speedup': speedup,
        })

        print(
            f'N={N:>10,} | run={run} | '
            f'CPU={tempo_cpu:>10.6f} ms | '
            f'GPU={tempo_gpu:>10.6f} ms | '
            f'speedup={speedup:>8.3f}x'
        )

df = pd.DataFrame(resultados)
df.to_csv(RESULTADOS_DIR / 'tempos.csv', index=False)

print('\nArquivo salvo:', RESULTADOS_DIR / 'tempos.csv')
display(df)


## 6. Calcular médias e speedup

A tabela abaixo resume a média das 5 execuções por tamanho de vetor.


In [ ]:
medias = (
    df.groupby('N')
      .agg(
          tempo_cpu_ms_medio=('tempo_cpu_ms', 'mean'),
          tempo_cpu_ms_desvio=('tempo_cpu_ms', 'std'),
          tempo_gpu_ms_medio=('tempo_gpu_ms', 'mean'),
          tempo_gpu_ms_desvio=('tempo_gpu_ms', 'std'),
      )
      .reset_index()
)

medias['speedup_medio'] = medias['tempo_cpu_ms_medio'] / medias['tempo_gpu_ms_medio']
medias.to_csv(RESULTADOS_DIR / 'tempos_medios.csv', index=False)

print('Arquivo salvo:', RESULTADOS_DIR / 'tempos_medios.csv')
display(medias)


## 7. Gráfico 1 — Tempo CPU vs GPU

O eixo Y usa escala logarítmica para facilitar a comparação entre valores pequenos e grandes.


In [ ]:
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    medias['N'], medias['tempo_cpu_ms_medio'],
    marker='o', linewidth=2.5, markersize=8,
    label='CPU sequencial', color='#e74c3c'
)
ax.plot(
    medias['N'], medias['tempo_gpu_ms_medio'],
    marker='s', linewidth=2.5, markersize=8,
    label='GPU CUDA', color='#2ecc71'
)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Tamanho do vetor (N)', fontsize=12)
ax.set_ylabel('Tempo médio (ms)', fontsize=12)
ax.set_title('Tempo de execução: CPU vs GPU', fontsize=14, fontweight='bold')
ax.set_xticks(medias['N'])
ax.set_xticklabels([f'{n:,}'.replace(',', '.') for n in medias['N']])
ax.legend(fontsize=11)
ax.grid(True, which='both', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(RESULTADOS_DIR / 'grafico_tempos.png', dpi=150, bbox_inches='tight')
plt.show()

print('Gráfico salvo:', RESULTADOS_DIR / 'grafico_tempos.png')


## 8. Gráfico 2 — Speedup

O speedup mostra quantas vezes a versão GPU foi mais rápida que a versão CPU:

```text
speedup = tempo_CPU / tempo_GPU
```


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

cores = ['#e74c3c' if s < 1 else '#2ecc71' for s in medias['speedup_medio']]
barras = ax.bar(
    [str(n) for n in medias['N']],
    medias['speedup_medio'],
    color=cores,
    edgecolor='white',
    linewidth=1.5,
)

ax.axhline(y=1, color='gray', linestyle='--', linewidth=1.2, label='Speedup = 1 (empate)')

for barra, valor in zip(barras, medias['speedup_medio']):
    altura = barra.get_height()
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        altura,
        f'{valor:.2f}x',
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold',
    )

ax.set_xlabel('Tamanho do vetor (N)', fontsize=12)
ax.set_ylabel('Speedup médio (CPU/GPU)', fontsize=12)
ax.set_title('Speedup da GPU em relação à CPU', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(RESULTADOS_DIR / 'grafico_speedup.png', dpi=150, bbox_inches='tight')
plt.show()

print('Gráfico salvo:', RESULTADOS_DIR / 'grafico_speedup.png')


## 9. Arquivos gerados

Ao final da execução, os principais arquivos ficam em `resultados/`:

- `resultados/tempos.csv`
- `resultados/tempos_medios.csv`
- `resultados/grafico_tempos.png`
- `resultados/grafico_speedup.png`


In [ ]:
print('Arquivos em resultados/:')
for arquivo in sorted(RESULTADOS_DIR.iterdir()):
    print('-', arquivo)


---

## Conclusão esperada

Para vetores pequenos, a CPU pode vencer porque a GPU tem overhead de transferência de dados e lançamento da kernel. Para vetores grandes, a GPU tende a vencer porque milhares de threads trabalham em paralelo, como um exército de clones fazendo a mesma missão em partes diferentes do vetor.
